# 01 — Build the portable data bundle (research machine)

Run this **locally**, where the nflbdb warehouse and the Milestone 12 outputs exist (12.0 executed, 12F Part 3 sequence store built in FULL mode; 12C for the transfer target). It copies, byte for byte:

- the 12F sequence store (`X.npy`, `M.npy`, `meta.parquet`)
- the 12.0 fold manifest
- optional: officials + cohort link + 12C target (`athletes.parquet`), Milestone 06 TCN-B fold representations

Upload the output folder to Google Drive (`MyDrive/NFLBowl/bundles/m12f_v1`) or as a Kaggle dataset. Nothing is recomputed.

In [ ]:
from nflbdb.data.config import Phase01Config, configure_runtime

cfg = Phase01Config.load()
configure_runtime(cfg)

from nflbdb.research.advanced_movement import config as amc
from nflbdb.research.advanced_movement.artifacts import ResearchStore
from nflbdb.research.advanced_movement.m11 import config as m11c
from nflbdb.research.advanced_movement.m11 import sources as src
from nflbdb.research.advanced_movement.m12 import config as m12c
from nflbdb.research.advanced_movement.m12 import foundation as fd

controls = amc.NotebookControls(run_mode="full", seed=20261015).validate()     # export the FULL store, never smoke
store = ResearchStore(cfg, controls)
C = m12c.M12Config().for_mode(controls)
S = src.M11Sources(cfg, store, m11c.M11Config().for_mode(controls))
D = fd.M12Data(store, C, S)

In [ ]:
from nflbowl.data.export import export_research_bundle

OUT = cfg.paths.project_root / "bundles" / "m12f_v1"      # git-ignored
bundle_json = export_research_bundle(store, D, OUT, include_transfer=True, include_tcn=True)
bundle_json

In [ ]:
# verify the copy exactly as a GPU session will read it (sha256 of every file)
from nflbowl.data.loading import open_bundle

b = open_bundle(OUT, verify=True, outer_folds=C.outer_folds)
b.summary()

Next: upload `bundles/m12f_v1/` to Drive and open `00_colab_quickstart.ipynb` (or `02_train_gpu.ipynb`) in Colab.